# Clustering of Remote-Sensing (ДЗЗ) Images — ГРУПА ВИМОГ 2

Cluster a set of Earth-surface remote-sensing images (≥ 10 instances) by visual
features (e.g. crops, forests, buildings) using **classical ML only — no deep learning**.

Pipeline: **load → analyze/enhance → feature extraction → clustering → visualize → save results**.

Data sources (download ≥ 10 images into `../data/raw/`): Copernicus Browser, NASA Earthdata,
EOS LandViewer, Kaggle, Sentinel Hub. See `task4.pdf` for the full list.

## 1. Imports & configuration

In [ ]:
from pathlib import Path

import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans, AgglomerativeClustering
from sklearn.metrics import silhouette_score
from scipy.cluster.hierarchy import dendrogram, linkage

DATA_DIR = Path('data/raw')
RESULTS_DIR = Path('results')
IMG_EXTS = {'.png', '.jpg', '.jpeg', '.tif', '.tiff', '.bmp'}
%matplotlib inline

## 2. Load the dataset

In [ ]:
# Preview the dataset
cols = 5
rows = int(np.ceil(len(dataset) / cols))
fig, axes = plt.subplots(rows, cols, figsize=(3 * cols, 3 * rows))
for ax, (name, img) in zip(np.ravel(axes), dataset):
    ax.imshow(img)
    ax.set_title(name, fontsize=8)
    ax.axis('off')
for ax in np.ravel(axes)[len(dataset):]:
    ax.axis('off')
plt.tight_layout()
plt.show()

In [ ]:
def enhance(img):
    """Improve image quality before feature extraction.

    Applies a light denoising filter and CLAHE contrast correction on the
    luminance channel (histogram correction). Returns an RGB uint8 image.
    """
    blurred = cv2.GaussianBlur(img, (3, 3), 0)
    lab = cv2.cvtColor(blurred, cv2.COLOR_RGB2LAB)
    l, a, b = cv2.split(lab)
    clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8))
    l = clahe.apply(l)
    return cv2.cvtColor(cv2.merge((l, a, b)), cv2.COLOR_LAB2RGB)


# Example: brightness histogram of the first image, before vs after enhancement
name, img = dataset[0]
enh = enhance(img)
fig, axes = plt.subplots(1, 2, figsize=(10, 3))
for ax, im, title in zip(axes, (img, enh), ('original', 'enhanced')):
    gray = cv2.cvtColor(im, cv2.COLOR_RGB2GRAY)
    ax.hist(gray.ravel(), bins=256, range=(0, 255), color='steelblue')
    ax.set_title(f'{title}: brightness histogram')
plt.tight_layout()
plt.show()

## 4. Feature extraction
Hand-crafted descriptors per image (no deep learning): color histograms + color moments.

In [ ]:
def extract_features(img, bins=8):
    """Build a fixed-length feature vector describing image content.

    Combines a normalized HSV color histogram with per-channel color moments
    (mean, std). These capture the dominant land-cover tones (vegetation,
    bare soil, water, built-up) without any learned representation.
    """
    hsv = cv2.cvtColor(img, cv2.COLOR_RGB2HSV)
    hist = cv2.calcHist([hsv], [0, 1, 2], None, [bins, bins, bins],
                        [0, 180, 0, 256, 0, 256])
    hist = cv2.normalize(hist, hist).flatten()
    moments = np.concatenate([hsv.mean(axis=(0, 1)), hsv.std(axis=(0, 1))])
    return np.concatenate([hist, moments]).astype(np.float32)


names = [n for n, _ in dataset]
X = np.vstack([extract_features(enhance(img)) for _, img in dataset])
X = StandardScaler().fit_transform(X)
print('Feature matrix:', X.shape)

## 5. Clustering
K-Means and hierarchical (agglomerative) clustering. Pick `k` via silhouette score.

In [ ]:
# Choose the number of clusters by silhouette score
candidates = range(2, min(8, len(dataset)))
scores = {k: silhouette_score(X, KMeans(n_clusters=k, n_init=10, random_state=42).fit_predict(X))
          for k in candidates}
best_k = max(scores, key=scores.get)
print('silhouette by k:', {k: round(v, 3) for k, v in scores.items()})
print('best_k =', best_k)

kmeans_labels = KMeans(n_clusters=best_k, n_init=10, random_state=42).fit_predict(X)
hier_labels = AgglomerativeClustering(n_clusters=best_k).fit_predict(X)

In [ ]:
# Hierarchical structure as a dendrogram
plt.figure(figsize=(10, 4))
dendrogram(linkage(X, method='ward'), labels=names, leaf_rotation=90)
plt.title('Ward hierarchical clustering')
plt.tight_layout()
plt.savefig(FIG_DIR / 'dendrogram.png', dpi=150)
plt.show()

## 6. Visualize & save results

In [ ]:
def show_clusters(labels, title, fname):
    """Plot images grouped by cluster label and save the figure."""
    order = np.argsort(labels)
    cols = 5
    rows = int(np.ceil(len(dataset) / cols))
    fig, axes = plt.subplots(rows, cols, figsize=(3 * cols, 3 * rows))
    for ax, idx in zip(np.ravel(axes), order):
        ax.imshow(dataset[idx][1])
        ax.set_title(f'cluster {labels[idx]}\n{names[idx]}', fontsize=8)
        ax.axis('off')
    for ax in np.ravel(axes)[len(dataset):]:
        ax.axis('off')
    fig.suptitle(title)
    plt.tight_layout()
    plt.savefig(FIG_DIR / fname, dpi=150)
    plt.show()


show_clusters(kmeans_labels, 'K-Means clusters', 'clusters_kmeans.png')
show_clusters(hier_labels, 'Hierarchical clusters', 'clusters_hierarchical.png')

In [ ]:
# Persist the cluster assignment table
df = pd.DataFrame({'image': names, 'kmeans': kmeans_labels, 'hierarchical': hier_labels})
df.to_csv(RESULTS_DIR / 'cluster_assignments.csv', index=False)
df